# A5 polygon polyfill (BFS)

Step-by-step BFS for `polygon2a5` (bbox expansion per part, then predicate filter).

**Neighbor count:** `grid_disk_vertex` + `uncompact` yields **7–8** cells around the seed.

Input: [`multipolygon.geojson`](https://raw.githubusercontent.com/opengeoshub/vopendata/main/shape/multipolygon.geojson) — **12 features**, **13 polygon parts**. **A5 resolution** is shown on every frame.

## Install necessary packages

In [1]:
%pip install vgrid geopandas matplotlib imageio pillow
# optional for MP4:
%pip install imageio-ffmpeg

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
"""Step-by-step polygon2a5 animation (multipolygon.geojson)."""
from collections import deque
from pathlib import Path

import a5
import geopandas as gpd
import imageio.v2 as imageio
import matplotlib.pyplot as plt
from matplotlib.collections import PatchCollection
from matplotlib.patches import Polygon as MplPolygon
from shapely.geometry import MultiPolygon, box

from vgrid.conversion.dggs2geo.a52geo import a52geo_u64
from vgrid.conversion.vector2dggs.vector2a5 import polygon2a5
from vgrid.utils.geometry import check_predicate

URL = "https://raw.githubusercontent.com/opengeoshub/vopendata/main/shape/multipolygon.geojson"
RESOLUTION = 12
PREDICATE = "intersects"
OUT_GIF = "polygon2a5.gif"
OUT_MP4 = "polygon2a5.mp4"
FRAME_EVERY_N_BFS = 8
DPI = 120
PART_COLORS = ["#1f4e79", "#c55a11", "#2e7d32", "#b71c1c", "#6a1b9a", "#4e342e", "#00695c", "#5d4037"]


def cell_patches(cell_polys, facecolor, edgecolor, alpha=0.55, lw=0.4):
    patches = []
    for poly in cell_polys:
        if poly is None or poly.is_empty:
            continue
        patches.append(MplPolygon(list(poly.exterior.coords), closed=True))
    return PatchCollection(
        patches, facecolor=facecolor, edgecolor=edgecolor, alpha=alpha, linewidths=lw
    )


def polygons_from_feature(feature):
    if feature.geom_type == "Polygon":
        return [feature]
    if feature.geom_type == "MultiPolygon":
        return list(feature.geoms)
    return []


def polygons_from_gdf(gdf):
    parts = []
    for geom in gdf.geometry:
        if geom is None or geom.is_empty:
            continue
        parts.extend(polygons_from_feature(geom))
    return parts


def feature_from_gdf(gdf):
    parts = polygons_from_gdf(gdf)
    if not parts:
        raise ValueError("No polygon geometries found in input GeoJSON")
    if len(parts) == 1:
        return parts[0]
    return MultiPolygon(parts)


def part_color(part_index):
    return PART_COLORS[part_index % len(PART_COLORS)]


def render_frame(
    parts,
    bbox,
    seed_poly,
    active_cells,
    title,
    path,
    resolution,
    current_poly=None,
    active_part=None,
):
    fig, ax = plt.subplots(figsize=(8, 8))
    union = MultiPolygon(parts) if len(parts) > 1 else parts[0]
    minx, miny, maxx, maxy = union.bounds
    pad = max(maxx - minx, maxy - miny) * 0.08 or 0.01
    ax.set_xlim(minx - pad, maxx + pad)
    ax.set_ylim(miny - pad, maxy + pad)

    for i, poly in enumerate(parts):
        color = part_color(i)
        lw = 3.0 if active_part == i else 1.8
        alpha = 1.0 if active_part is None or active_part == i else 0.45
        gpd.GeoSeries([poly]).plot(
            ax=ax, facecolor="none", edgecolor=color, lw=lw, alpha=alpha
        )
    if bbox is not None:
        gpd.GeoSeries([bbox]).plot(
            ax=ax, facecolor="none", edgecolor="#ff7f0e", lw=1.5, linestyle="--"
        )

    visited = list(active_cells) if active_cells else []
    if current_poly is not None:
        visited = [
            p
            for p in visited
            if p is not current_poly and not p.equals(current_poly)
        ]
    if visited:
        ax.add_collection(cell_patches(visited, "#2ca02c", "#1a5f1a", alpha=0.45))
    if seed_poly is not None and (
        current_poly is None
        or (seed_poly is not current_poly and not seed_poly.equals(current_poly))
    ):
        ax.add_collection(cell_patches([seed_poly], "#d62728", "#8b0000", alpha=0.7))
    if current_poly is not None:
        ax.add_collection(
            cell_patches([current_poly], "#ffcc00", "#cc8800", alpha=0.9, lw=2.5)
        )
    ax.plot([], [], color="#ffcc00", lw=4, label="current cid (deque)")
    ax.plot([], [], color="#2ca02c", lw=4, label="visited")
    ax.plot([], [], color="#d62728", lw=4, label="seed")
    ax.legend(loc="upper right", fontsize=8)
    ax.text(
        0.02,
        0.98,
        f"A5 resolution: {resolution}",
        transform=ax.transAxes,
        fontsize=9,
        va="top",
        ha="left",
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.9),
        zorder=6,
    )
    ax.set_title(title)
    ax.set_aspect("equal")
    ax.grid(True, alpha=0.25)
    fig.subplots_adjust(left=0.08, right=0.92, top=0.92, bottom=0.08)
    fig.savefig(path, dpi=DPI, facecolor="white")
    plt.close(fig)


def polygon2a5_with_frames(parts, feature, resolution, predicate, frame_dir):
    frame_dir.mkdir(parents=True, exist_ok=True)
    frames = []
    idx = 0
    merged_ids = []
    merged_polys = []
    seen_u64 = set()
    accumulated = []

    def snap(title, bbox=None, seed=None, active=None, current=None, active_part=None):
        nonlocal idx
        p = frame_dir / f"frame_{idx:04d}.png"
        render_frame(
            parts,
            bbox,
            seed,
            active if active is not None else accumulated,
            title,
            p,
            resolution,
            current_poly=current,
            active_part=active_part,
        )
        frames.append(p)
        idx += 1

    n_parts = len(parts)
    snap(
        f"1. Input res {resolution} ({n_parts} polygon part{'s' if n_parts != 1 else ''})"
    )
    snap("2. All parts (distinct colors)")

    for part_i, polygon in enumerate(parts, start=1):
        min_lng, min_lat, max_lng, max_lat = polygon.bounds
        bbox = box(min_lng, min_lat, max_lng, max_lat)
        lon, lat = bbox.centroid.x, bbox.centroid.y
        seed_id = a5.lonlat_to_cell((lon, lat), resolution)
        seed_poly = a52geo_u64(seed_id)
        part_idx = part_i - 1

        snap(
            f"3. Part {part_i}/{n_parts}: seed at bbox centroid",
            bbox=bbox,
            seed=seed_poly,
            active_part=part_idx,
        )

        neighbor_polys = []
        for nid in a5.uncompact(a5.grid_disk_vertex(seed_id, 1), resolution):
            if nid == seed_id:
                continue
            npoly = a52geo_u64(nid)
            if npoly is not None and not npoly.is_empty:
                neighbor_polys.append(npoly)
        snap(
            f"3b. Part {part_i} seed neighbors (grid_disk_vertex, {len(neighbor_polys)} cells)",
            bbox=bbox,
            seed=seed_poly,
            active=neighbor_polys,
            active_part=part_idx,
        )

        if seed_poly.contains(bbox):
            if seed_id not in seen_u64:
                seen_u64.add(seed_id)
                merged_ids.append(seed_id)
                merged_polys.append(seed_poly)
                accumulated.append(seed_poly)
            snap(
                f"4. Part {part_i} seed covers bbox — single cell",
                bbox=bbox,
                seed=seed_poly,
                active=[seed_poly],
                active_part=part_idx,
            )
            continue

        intersecting = {}
        covered = set()
        queue = deque([seed_id])
        bfs_step = 0
        while queue:
            cid = queue.popleft()
            if cid in covered:
                continue
            covered.add(cid)
            cell_poly = a52geo_u64(cid)
            if cell_poly is None or cell_poly.is_empty:
                continue
            if cell_poly.intersects(bbox):
                intersecting[cid] = cell_poly
                for nid in a5.uncompact(a5.grid_disk_vertex(cid, 1), resolution):
                    if nid not in covered:
                        queue.append(nid)
                bfs_step += 1
                if bfs_step % FRAME_EVERY_N_BFS == 0:
                    snap(
                        f"4. Part {part_i} BFS step {bfs_step}: {a5.u64_to_hex(cid)} ({len(intersecting)} cells)",
                        bbox=bbox,
                        seed=seed_poly,
                        active=list(intersecting.values()),
                        current=cell_poly,
                        active_part=part_idx,
                    )
        snap(
            f"5. Part {part_i} BFS complete ({len(intersecting)} candidates)",
            bbox=bbox,
            seed=seed_poly,
            active=list(intersecting.values()),
            active_part=part_idx,
        )

        part_final = []
        for cid, cell_poly in intersecting.items():
            if check_predicate(cell_poly, polygon, predicate):
                part_final.append((cid, cell_poly))
                if cid not in seen_u64:
                    seen_u64.add(cid)
                    merged_ids.append(cid)
                    merged_polys.append(cell_poly)
                    accumulated.append(cell_poly)
        snap(
            f"6. Part {part_i} after predicate '{predicate}' ({len(part_final)} cells)",
            bbox=bbox,
            seed=seed_poly,
            active=[p for _, p in part_final],
            active_part=part_idx,
        )

    snap(
        f"7. Merged result ({len(merged_ids)} cells across {n_parts} parts)",
        active=merged_polys,
    )

    from_poly = []
    for part in parts:
        rows = polygon2a5(part, resolution, predicate=predicate)
        from_poly.extend(row["a5"] for row in rows)
    our_hex = [a5.u64_to_hex(c) for c in merged_ids]
    if set(from_poly) != set(our_hex):
        print(
            "Warning: cell set differs from polygon2a5:",
            len(our_hex),
            "vs",
            len(set(from_poly)),
        )

    return frames, merged_ids


def main():
    gdf = gpd.read_file(URL)
    parts = polygons_from_gdf(gdf)
    feature = feature_from_gdf(gdf)
    print(f"Loaded {len(gdf)} feature(s), {len(parts)} polygon part(s)")

    frame_dir = Path("_polygon2a5_frames")
    frames, cell_ids = polygon2a5_with_frames(
        parts, feature, RESOLUTION, PREDICATE, frame_dir
    )
    imageio.mimsave(OUT_GIF, [imageio.imread(f) for f in frames], duration=0.9)
    print(f"Wrote {OUT_GIF} ({len(frames)} frames, {len(cell_ids)} final cells)")
    try:
        writer = imageio.get_writer(OUT_MP4, fps=1.2)
        for f in frames:
            writer.append_data(imageio.imread(f))
        writer.close()
        print(f"Wrote {OUT_MP4}")
    except Exception as e:
        print(f"MP4 skipped ({e}). GIF is enough.")


if __name__ == "__main__":
    main()
